In [1]:
!pip install -U pandas scikit-learn
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import joblib
SEED = 42


  You can safely remove it manually.


   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.0 MB 10.3 MB/s eta 0:00:01
   ------------------ --------------------- 5.0/11.0 MB 12.9 MB/s eta 0:00:01
   ------------------------------ --------- 8.4/11.0 MB 14.2 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 13.6 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 13.1 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.2
    Uninstalling pandas-2.3.2:
      Successfully uninstalled pandas-2.3.2


In [7]:
df = pd.read_csv(r"C:\Users\anshu\Documents\PROJECTS\fake-news-spam-detector\data\raw\SMSSpamCollection",
                 sep="\t", header=None, names=["label","text"])
print("Shape:", df.shape); print(df["label"].value_counts()); df.head()


Shape: (5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
train_pool, test_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED, shuffle=True)
train_df, val_df  = train_test_split(train_pool, test_size=0.125, stratify=train_pool["label"], random_state=SEED, shuffle=True)
for name, d in [("train",train_df),("val",val_df),("test",test_df)]:
    print(name, d.shape, d["label"].value_counts(normalize=True))


train (3899, 2) label
ham     0.865863
spam    0.134137
Name: proportion, dtype: float64
val (558, 2) label
ham     0.865591
spam    0.134409
Name: proportion, dtype: float64
test (1115, 2) label
ham     0.866368
spam    0.133632
Name: proportion, dtype: float64


In [9]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2, lowercase=True, strip_accents="unicode")
X_tr = vectorizer.fit_transform(train_df["text"])
X_va = vectorizer.transform(val_df["text"])
X_te = vectorizer.transform(test_df["text"])
y_tr, y_va, y_te = train_df["label"], val_df["label"], test_df["label"]
X_tr.shape


(3899, 10780)

In [10]:
clf = LogisticRegression(solver="liblinear", max_iter=1000, random_state=SEED)
clf.fit(X_tr, y_tr)
"trained"


'trained'

In [11]:
def eval_split(X, y, model, pos_label="spam"):
    y_hat = model.predict(X)
    acc = accuracy_score(y, y_hat)
    f1  = f1_score(y, y_hat, pos_label=pos_label, average="binary")
    if hasattr(model,"predict_proba"):
        p = model.predict_proba(X)[:,1]
    else:
        s = model.decision_function(X).reshape(-1,1)
        p = MinMaxScaler().fit_transform(s).ravel()
    y_bin = (y==pos_label).astype(int)
    auc = roc_auc_score(y_bin, p)
    rep = classification_report(y, y_hat, digits=4)
    return {"accuracy":acc,"f1":f1,"roc_auc":auc,"report":rep}, y_hat


In [12]:
val_metrics, val_pred = eval_split(X_va, y_va, clf)
print("VAL:", {k:round(v,4) for k,v in val_metrics.items() if k!="report"}); print(val_metrics["report"])
test_metrics, test_pred = eval_split(X_te, y_te, clf)
print("TEST:", {k:round(v,4) for k,v in test_metrics.items() if k!="report"}); print(test_metrics["report"])


VAL: {'accuracy': 0.9624, 'f1': 0.8372, 'roc_auc': 0.9808}
              precision    recall  f1-score   support

         ham     0.9583    1.0000    0.9787       483
        spam     1.0000    0.7200    0.8372        75

    accuracy                         0.9624       558
   macro avg     0.9792    0.8600    0.9080       558
weighted avg     0.9639    0.9624    0.9597       558

TEST: {'accuracy': 0.9686, 'f1': 0.8669, 'roc_auc': 0.9887}
              precision    recall  f1-score   support

         ham     0.9650    1.0000    0.9822       966
        spam     1.0000    0.7651    0.8669       149

    accuracy                         0.9686      1115
   macro avg     0.9825    0.8826    0.9246      1115
weighted avg     0.9697    0.9686    0.9668      1115



In [13]:
import pandas as pd, os
os.makedirs("errors", exist_ok=True); os.makedirs("models", exist_ok=True)
errs = test_df.loc[test_pred != y_te].copy(); errs["pred"] = test_pred[test_pred != y_te]
errs.to_csv("errors/misclassified_test.csv", index=False)
joblib.dump(vectorizer, "models/tfidf_vectorizer.joblib"); joblib.dump(clf,"models/baseline_logreg.joblib")
"Saved errors and model artifacts"


'Saved errors and model artifacts'